
# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [3]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np
from scipy import sparse


# Amelia -> load data from github repo
!git clone https://github.com/mk654/SML_PG60
repo_dir = Path("/content/SML_PG60")

flu_mat = repo_dir / "data" / "matraw" / "influenza_outbreak_dataset.mat" # access influenza dataset from gitrepo
fludata = loadmat(flu_mat)



# url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat" # old access -> directory = main
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat" # old access -> directory = main/data


Cloning into 'SML_PG60'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 279 (delta 65), reused 1 (delta 1), pack-reused 159 (from 1)
Receiving objects: 100% (279/279), 41.65 MiB | 27.71 MiB/s, done.
Resolving deltas: 100% (114/114), done.


In [1]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CATBOOST_TASK_TYPE = "GPU" if DEVICE == "cuda" else "CPU"

print(f"Using PyTorch device: {DEVICE}")
print(f"Using CatBoost task type: {CATBOOST_TASK_TYPE}")

Using PyTorch device: cuda
Using CatBoost task type: GPU


### 0.a) inspecting data

*Amelia*


#### *Results*
| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

in sum:
- 48 training feature matrices, one per location
- 48 testing feature matrices, one per location
- 48 training label vectors
- 48 testing label vectors
- 48 location names/IDs
- 545 keyword feature names



*Code*


```
rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)
```

In [ ]:
# delete this cell before submission!!

rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)

           name outer_dtype outer_shape inner_type  inner_shape
0      flu_X_tr      object     (1, 48)  csc_array  (1095, 545)
1      flu_X_te      object     (1, 48)  csc_array   (485, 545)
2      flu_Y_tr      object     (1, 48)    ndarray    (1095, 1)
3      flu_Y_te      object     (1, 48)    ndarray     (485, 1)
4      flu_locs      object     (1, 48)    ndarray         (1,)
5  flu_keywords      object    (1, 525)    ndarray         (1,)


### 0.b) convert .mat files to .csv
Converts influenza_outbreak_dataset.mat to csv

github directory = ```SML_PG60/data/processed/flu_csv```

----
*Note that these csv files are only processed in the sense that they have been converted from .mat to .csv; no further processing has yet taken place.*

### Amelia's conversion
- conversion will produce 48 separate folders aligning with the 48 folds. Each folder will contain:
  1. X_train.csv
  2. X_test.csv
  3. y_train.csv
  4. y_test.csv
  * *note that influenza_outbreak_dataset.mat contains 48 folds (test/train splits). Each fold contains its own X & y train and X & y test. Hence, the data is split and converted as cleanly as possible to avoid errors that may come with combining folds*
- Additionally, the conversion will also produce:
  1. keywords.csv
  2. locs.csv

---


The converted files can be manually downloaded from this notebook by copy/pasting & running the following code:

*this isn't currently working :(*
```
from google.colab import files
!zip -r ak_flucsv.zip /content/SML_PG60/data/processed/flu_csv/"ak_flucsv"
files.download("ak_flucsv.zip")
```

In [4]:
# Amelia -> convert influenza_outbreak_dataset.mat to .csv file
flu_out = repo_dir / "data" / "processed" / "flu_csv" / "ak_flucsv"
flu_out.mkdir(parents=True, exist_ok=True)

X_tr = fludata["flu_X_tr"]
X_te = fludata["flu_X_te"]
y_tr = fludata["flu_Y_tr"]
y_te = fludata["flu_Y_te"]

n_folds = X_tr.shape[1]

for i in range(n_folds):
    fold_dir = flu_out / f"fold_{i:02d}"
    fold_dir.mkdir(exist_ok=True)

    Xtr = X_tr[0, i].toarray() # some data stored as sparse matrix, convert to dense
    Xte = X_te[0, i].toarray()
    ytr = y_tr[0, i].ravel()
    yte = y_te[0, i].ravel()

    pd.DataFrame(Xtr).to_csv(fold_dir / "X_train.csv", index=False)
    pd.DataFrame(Xte).to_csv(fold_dir / "X_test.csv", index=False)
    pd.DataFrame(ytr).to_csv(fold_dir / "y_train.csv", index=False)
    pd.DataFrame(yte).to_csv(fold_dir / "y_test.csv", index=False)

    print(f"Saved fold {i}")

keywords = fludata["flu_keywords"]
keywords_list = [str(k[0]) for k in keywords.ravel()]
pd.DataFrame(keywords_list, columns=["keyword"]) \
  .to_csv(flu_out / "keywords.csv", index=False)

locs = fludata["flu_locs"]
locs_list = [str(l[0]) for l in locs.ravel()]
pd.DataFrame(locs_list, columns=["location"]) \
  .to_csv(flu_out / "locs.csv", index=False)

Saved fold 0
Saved fold 1
Saved fold 2
Saved fold 3
Saved fold 4
Saved fold 5
Saved fold 6
Saved fold 7
Saved fold 8
Saved fold 9
Saved fold 10
Saved fold 11
Saved fold 12
Saved fold 13
Saved fold 14
Saved fold 15
Saved fold 16
Saved fold 17
Saved fold 18
Saved fold 19
Saved fold 20
Saved fold 21
Saved fold 22
Saved fold 23
Saved fold 24
Saved fold 25
Saved fold 26
Saved fold 27
Saved fold 28
Saved fold 29
Saved fold 30
Saved fold 31
Saved fold 32
Saved fold 33
Saved fold 34
Saved fold 35
Saved fold 36
Saved fold 37
Saved fold 38
Saved fold 39
Saved fold 40
Saved fold 41
Saved fold 42
Saved fold 43
Saved fold 44
Saved fold 45
Saved fold 46
Saved fold 47


#### Amelia → Combine outputs
Combination keeps the original test/train split from influenza_outbreak_dataset.csv:
1. ```train_all_locs.csv```
2. ```test_all_locs.csv```

Additionally, every single feature within X-matrices are also kept.

---
This folder can be zipped and downloaded by running the following:
```
from google.colab import files
!zip -r ak_flucsv_combined.zip /content/SML_PG60/data/processed/"ak_flucsv_combined"
files.download("ak_flucsv_combined.zip")
```



In [5]:

combined_out = repo_dir / "data" / "processed" / "ak_flucsv_combined"
combined_out.mkdir(parents=True, exist_ok=True)

keywords = pd.read_csv(flu_out / "keywords.csv")["keyword"].tolist()
locs = pd.read_csv(flu_out / "locs.csv")["location"].tolist()

train_parts = []
test_parts = []

for i, loc in enumerate(locs):
    fold_dir = flu_out / f"fold_{i:02d}"

    X_train = pd.read_csv(fold_dir / "X_train.csv")
    y_train = pd.read_csv(fold_dir / "y_train.csv")
    X_test = pd.read_csv(fold_dir / "X_test.csv")
    y_test = pd.read_csv(fold_dir / "y_test.csv")

    n_features = X_train.shape[1]
    n_keywords = len(keywords)
    extra_cols = [f"extra_feature{j}" for j in range(n_features - n_keywords)] #keep all features
    feature_cols = keywords + extra_cols

    X_train.columns = feature_cols
    X_test.columns = feature_cols

    train_df = X_train.copy()
    train_df["label"] = y_train.iloc[:, 0].values
    train_df.insert(0, "split", "train")
    train_df.insert(0, "fold_location", loc)
    train_df.insert(0, "fold", i)

    test_df = X_test.copy()
    test_df["label"] = y_test.iloc[:, 0].values
    test_df.insert(0, "split", "test")
    test_df.insert(0, "fold_location", loc)
    test_df.insert(0, "fold", i)

    train_parts.append(train_df)
    test_parts.append(test_df)

train_all = pd.concat(train_parts, ignore_index=True)
test_all = pd.concat(test_parts, ignore_index=True)

train_all.to_csv(combined_out / "train_all_locs.csv", index=False)
test_all.to_csv(combined_out / "test_all_locs.csv", index=False)

print("saved combined csvs.")
print("train shape:", train_all.shape)
print("test shape:", test_all.shape)

saved combined csvs.
train shape: (52560, 549)
test shape: (23280, 549)


### Songhao's conversion
- combines all information from influenza_outbreak_dataset.mat into  ```influenza_outbreak_long.csv``` (>150MB)
- original X matrices within .mat file contain 545 features, however, only 525 named keyword features exist
  * thus ```influenza_outbreak_long.csv``` drops last 20 features within original X matrices
- adds column "time_index" which counts rows in each train/test split per location

In [6]:
# (Songhao) convert loaded .mat variables to CSV
sg_dir = repo_dir / "data" / "processed" / "flu_csv" / "sg_flucsv"
sg_dir.mkdir(parents=True, exist_ok=True)

csv_output_path = sg_dir / "influenza_outbreak_long.csv"
keywords_output_path = sg_dir / "flu_keywords.csv"
locations_output_path = sg_dir / "flu_locations.csv"

def extract_matlab_string_array(arr):
    values = []
    for item in arr.flatten():
        if isinstance(item, np.ndarray):
            if item.size == 1:
                values.append(str(item.item()).strip())
            else:
                values.append("".join(item.astype(str).flatten()).strip())
        else:
            values.append(str(item).strip())
    return values


def make_safe_unique_names(names):
    safe_names = []
    used = {}

    for name in names:
        clean = (
            str(name)
            .strip()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("/", "_")
            .replace("(", "")
            .replace(")", "")
        )

        if clean == "":
            clean = "unnamed_feature"

        if clean in used:
            used[clean] += 1
            clean = f"{clean}_{used[clean]}"
        else:
            used[clean] = 0

        safe_names.append(clean)

    return safe_names

#-------------------------------#
locs = fludata["flu_locs"]
keywrds = fludata["flu_keywords"]
X_tr = fludata["flu_X_tr"]
y_tr = fludata["flu_Y_tr"]
X_te = fludata["flu_X_te"]
y_te = fludata["flu_Y_te"]
#-------------------------------#

locations = extract_matlab_string_array(locs)
keywords = extract_matlab_string_array(keywrds)
feature_names = make_safe_unique_names(keywords)

all_parts = []

for i, location in enumerate(locations):
    for split_name, X_cell, y_cell in [
        ("train", X_tr, y_tr),
        ("test", X_te, y_te),
    ]:
        X = X_cell[0, i]

        if sparse.issparse(X):
            X = X.toarray()
        else:
            X = np.asarray(X)

        # Keep only the 525 named keyword features --> original X matrices contain 545 features (final 20 columns are dropped)
        X = X[:, :len(feature_names)]

        y = np.asarray(y_cell[0, i]).reshape(-1).astype(int)

        df_part = pd.DataFrame(X, columns=feature_names)
        df_part.insert(0, "location", location)
        df_part.insert(1, "split", split_name)
        df_part.insert(2, "time_index", np.arange(len(y)))
        df_part["label"] = y

        all_parts.append(df_part)

df = pd.concat(all_parts, ignore_index=True)

df.to_csv(csv_output_path, index=False)
pd.DataFrame({"keyword": keywords}).to_csv(keywords_output_path, index=False)
pd.DataFrame({"location": locations}).to_csv(locations_output_path, index=False)

print("Saved main CSV to:", csv_output_path)
print("Saved keywords CSV to:", keywords_output_path)
print("Saved locations CSV to:", locations_output_path)

print("\nData shape:", df.shape)
print("\nSplit counts:")
print(df["split"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts())

df.head()

Saved main CSV to: /content/SML_PG60/data/processed/flu_csv/sg_flucsv/influenza_outbreak_long.csv
Saved keywords CSV to: /content/SML_PG60/data/processed/flu_csv/sg_flucsv/flu_keywords.csv
Saved locations CSV to: /content/SML_PG60/data/processed/flu_csv/sg_flucsv/flu_locations.csv

Data shape: (75840, 529)

Split counts:
split
train    52560
test     23280
Name: count, dtype: int64

Label distribution:
label
0    70653
1     5187
Name: count, dtype: int64


,location,split,time_index,flu,swine,stomach,symptoms,virus,bug,strep,...,tests,thinks,ankle,work,hand,complications,children,start,aja,label
0,wyoming,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,wyoming,train,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,wyoming,train,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,wyoming,train,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,wyoming,train,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
#Lachlan -> parameter metrics
import torch
import torch.nn as nn

def accuracy_score(preds, y):
  correct = (preds == y).sum()
  return correct / len(y)

def precision_score(preds, y): #preds and y are arrays of 0 and 1
  tp = (preds * y).sum() #Only indicies where preds and y are 1 will be counted
  pred_positives = (preds == 1).sum()
  return tp / (pred_positives + 1e-7) #To prevent div 0 problems

def recall_score(preds, y): #preds and y are arrays of 0 and 1
  tp = (preds * y).sum() #Only indicies where preds and y are 1 will be counted
  real_positives = (y == 1).sum()
  return tp / (real_positives + 1e-7) #To prevent div 0 problems

def f1_score(preds, y):
    prec = precision_score(preds, y)
    rec = recall_score(preds, y)
    return 2 * (prec * rec) / (prec + rec + 1e-7)



In [ ]:
#Lachlan -> basic logistic regression model
class LogisticRegressionModel(nn.Module): #From tute
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

def train_model(model_class, input_dim, criterion_fn, lr, momentum, X_train, y_train, X_eval, y_eval, epochs=50, batch_size=32):
    model = model_class(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #Convert training and evaluation arrays to tensors
    Xt = torch.tensor(X_train, dtype = torch.float32)
    yt = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
    Xe = torch.tensor(X_eval, dtype = torch.float32)

    #Set up training data loader
    dataset = torch.utils.data.TensorDataset(Xt, yt)
    train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle= True)

    for epoch in range(epochs):
      model.train()
      for batch_X, batch_y in train_loader:
          optimizer.zero_grad()
          predictions = model(batch_X)
          loss = criterion_fn(predictions, batch_y)
          loss.backward()
          optimizer.step()

    model.eval()
    with torch.no_grad():
      outputs = model(Xe) #Determine the raw probabilities
      preds = (torch.sigmoid(outputs) >= 0.5).float().numpy().flatten() #Convert to array of 0 and 1

    acc = accuracy_score(preds, y_eval)
    prec = precision_score(preds, y_eval)
    rec = recall_score(preds, y_eval)
    f1 = f1_score(preds, y_eval)

    return acc, prec, rec, f1


In [ ]:
#Lachlan -> Cross-validated logistic regression
from sklearn.model_selection import StratifiedKFold


#Inputs are np arrays X and y
def cross_val(k_folds, X, y, model, criterion, lr = 0.001, momentum = 1e-2):
  n_features = X.shape[1]
  skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
  fold_accuracies = []
  fold_precisions = []
  fold_recalls = []
  fold_f1 = []

  folds = skf.split(X, y) #Create training, validation and testing folds

  for fold, (train_idx, val_idx) in enumerate(folds):
      X_train, X_eval = X[train_idx], X[val_idx]
      y_train, y_eval = y[train_idx], y[val_idx]

      acc, prec, rec, f1 = train_model(model, n_features, criterion, lr, momentum, X_train, y_train, X_eval, y_eval, epochs=50)

      fold_accuracies.append(acc)
      fold_precisions.append(prec)
      fold_recalls.append(rec)
      fold_f1.append(f1)

  avg_acc = np.mean(fold_accuracies)
  avg_prec = np.mean(fold_precisions)
  avg_rec = np.mean(fold_recalls)
  avg_f1 = np.mean(fold_f1)

  return avg_acc, avg_prec, avg_rec, avg_f1


In [11]:
#Lachlan -> run baseline logistic regression

from sklearn.preprocessing import StandardScaler
csv_path = repo_dir / "data" / "processed" / "flu_csv" / "sg_flucsv" / "influenza_outbreak_long.csv"
df = pd.read_csv(csv_path)
df.drop(columns = ['location','split', 'time_index'], inplace = True) #Basic version
df['label'] = df['label'].astype(int) #Cast entries as ints

X_np = (df.drop(columns = ['label'])).values
y_np = df['label'].values


criterion = torch.nn.BCEWithLogitsLoss()

#Configure for cross-validation
k_folds = 5

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)
avg_acc,  avg_prec, avg_rec, avg_f1 = cross_val(k_folds, X_scaled, y_np, LogisticRegressionModel, criterion, 0.001, 0.01)

print(f"Averages: Accuracy {avg_acc} | Precision {avg_prec} | Recall {avg_rec} | F1 {avg_f1}")

Averages: Accuracy 0.9343486286919831 | Precision 0.5659968316916677 | Recall 0.17331880348308754 | F1 0.26509696580784137


## 2. CatBoost Model with From-Scratch Nested Cross-Validation

This section adds CatBoost as the complex model. The artificial `time_index` / `timestamp` columns are removed because they were added during preprocessing and do not represent real temporal information.

In [9]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [10]:
# Songhao -> CatBoost model

import numpy as np
import pandas as pd


from catboost import CatBoostClassifier

In [11]:
# Songhao -> Prepare features and labels

target_col = "label"

drop_cols = ["location", "split", "time_index", "timestamp"]
drop_cols = [col for col in drop_cols if col in df.columns]

keyword_cols = [
    col for col in df.columns
    if col not in drop_cols + [target_col]
]

X_raw = df[keyword_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df[target_col].astype(int).to_numpy()

# Non-temporal feature construction
X = X_raw.copy()
X["keyword_total_intensity"] = X_raw.sum(axis=1)
X["active_keyword_count"] = (X_raw > 0).sum(axis=1)
X["nonzero_keyword_ratio"] = X["active_keyword_count"] / len(keyword_cols)
X["max_keyword_value"] = X_raw.max(axis=1)
X["mean_keyword_value"] = X_raw.mean(axis=1)
X["std_keyword_value"] = X_raw.std(axis=1)
X["log_total_intensity"] = np.log1p(X["keyword_total_intensity"])

catboost_feature_columns = X.columns.tolist()

print("Dropped columns:", drop_cols)
print("Original keyword features:", len(keyword_cols))
print("Final feature shape:", X.shape)
print("Label distribution:")
print(pd.Series(y).value_counts())

Dropped columns: ['location', 'split', 'time_index']
Original keyword features: 525
Final feature shape: (75840, 532)
Label distribution:
0    70653
1     5187
Name: count, dtype: int64


In [12]:
#  Songhao -> Metrics and stratified k-fold cross-validation from scratch

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def make_stratified_folds(y, n_folds, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)

    folds = [[] for _ in range(n_folds)]

    for label in np.unique(y):
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)

        for i, idx in enumerate(label_indices):
            folds[i % n_folds].append(idx)

    return [np.array(fold, dtype=int) for fold in folds]


def get_train_indices(n_samples, test_indices):
    all_indices = np.arange(n_samples)
    return np.setdiff1d(all_indices, test_indices)


def summarise_results(results_df):
    summary_rows = []

    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = results_df[metric].to_numpy(dtype=float)

        summary_rows.append({
            "metric": metric,
            "mean": values.mean(),
            "std": values.std(ddof=1),
        })

    return pd.DataFrame(summary_rows)

In [13]:
# Songhao -> CatBoost with nested cross-validation
def train_catboost(X_train, y_train, depth, iterations=200, learning_rate=0.05, seed=42):
    model = CatBoostClassifier(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        loss_function="Logloss",
        eval_metric="F1",
        auto_class_weights="Balanced",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        task_type=CATBOOST_TASK_TYPE,
        devices="0" if CATBOOST_TASK_TYPE == "GPU" else None,
    )

    model.fit(X_train, y_train)
    return model


def tune_catboost_depth(X_train_outer, y_train_outer, depth_grid, inner_folds=3, seed=42):
    inner_fold_indices = make_stratified_folds(
        y_train_outer,
        n_folds=inner_folds,
        seed=seed,
    )

    tuning_results = []

    for depth in depth_grid:
        inner_scores = []

        for inner_fold_id, val_idx in enumerate(inner_fold_indices):
            train_idx = get_train_indices(len(y_train_outer), val_idx)

            X_inner_train = X_train_outer.iloc[train_idx]
            y_inner_train = y_train_outer[train_idx]
            X_inner_val = X_train_outer.iloc[val_idx]
            y_inner_val = y_train_outer[val_idx]

            model = train_catboost(
                X_inner_train,
                y_inner_train,
                depth=depth,
                iterations=120,
                learning_rate=0.05,
                seed=seed + inner_fold_id,
            )

            y_val_pred = model.predict(X_inner_val).astype(int)
            val_metrics = compute_metrics(y_inner_val, y_val_pred)
            inner_scores.append(val_metrics["f1"])

        tuning_results.append({
            "depth": depth,
            "mean_inner_f1": float(np.mean(inner_scores)),
            "std_inner_f1": float(np.std(inner_scores, ddof=1)),
        })

    tuning_df = pd.DataFrame(tuning_results)
    best_depth = int(
        tuning_df.sort_values("mean_inner_f1", ascending=False).iloc[0]["depth"]
    )

    return best_depth, tuning_df


outer_folds = make_stratified_folds(y, n_folds=10, seed=42)
depth_grid = [10, 12, 14]

catboost_outer_results = []
catboost_tuning_logs = []

for outer_fold_id, test_idx in enumerate(outer_folds, start=1):
    print(f"Running outer fold {outer_fold_id}/10")

    train_idx = get_train_indices(len(y), test_idx)

    X_outer_train = X.iloc[train_idx].reset_index(drop=True)
    y_outer_train = y[train_idx]
    X_outer_test = X.iloc[test_idx].reset_index(drop=True)
    y_outer_test = y[test_idx]

    best_depth, tuning_df = tune_catboost_depth(
        X_outer_train,
        y_outer_train,
        depth_grid=depth_grid,
        inner_folds=3,
        seed=100 + outer_fold_id,
    )

    tuning_df["outer_fold"] = outer_fold_id
    catboost_tuning_logs.append(tuning_df)

    final_model = train_catboost(
        X_outer_train,
        y_outer_train,
        depth=best_depth,
        iterations=200,
        learning_rate=0.05,
        seed=200 + outer_fold_id,
    )

    y_test_pred = final_model.predict(X_outer_test).astype(int)
    test_metrics = compute_metrics(y_outer_test, y_test_pred)

    test_metrics["outer_fold"] = outer_fold_id
    test_metrics["best_depth"] = best_depth

    catboost_outer_results.append(test_metrics)

catboost_results_df = pd.DataFrame(catboost_outer_results)
catboost_tuning_df = pd.concat(catboost_tuning_logs, ignore_index=True)
catboost_summary_df = summarise_results(catboost_results_df)

display(catboost_results_df)
display(catboost_summary_df)

Running outer fold 1/10
Running outer fold 2/10
Running outer fold 3/10
Running outer fold 4/10
Running outer fold 5/10
Running outer fold 6/10
Running outer fold 7/10
Running outer fold 8/10
Running outer fold 9/10
Running outer fold 10/10


,accuracy,precision,recall,f1,tp,tn,fp,fn,outer_fold,best_depth
0,0.882927,0.285714,0.473988,0.356522,246,6451,615,273,1,12
1,0.885300,0.299886,0.506744,0.376791,263,6452,614,256,2,12
2,0.887805,0.300481,0.481696,0.370096,250,6484,582,269,3,12
3,0.883966,0.282268,0.450867,0.347181,234,6470,595,285,4,12
4,0.891614,0.316808,0.504817,0.389302,262,6500,565,257,5,12
5,0.888845,0.308511,0.502890,0.382418,261,6480,585,258,6,12
6,0.888054,0.306338,0.502890,0.380744,261,6474,591,258,7,12
7,0.887512,0.303173,0.498069,0.376917,258,6472,593,260,8,12
8,0.877225,0.272827,0.478764,0.347582,248,6404,661,270,9,12
9,0.886325,0.294258,0.474903,0.363368,246,6475,590,272,10,12


,metric,mean,std
0,accuracy,0.885957,0.003951
1,precision,0.297026,0.013362
2,recall,0.487563,0.018405
3,f1,0.369092,0.014820


In [14]:
# Songhao -> Train final CatBoost model on the full influenza dataset

best_depth_overall = int(catboost_results_df["best_depth"].mode().iloc[0])

final_catboost_model = train_catboost(
    X,
    y,
    depth=best_depth_overall,
    iterations=200,
    learning_rate=0.05,
    seed=999,
)

print("Final CatBoost model trained.")
print("Selected depth:", best_depth_overall)

Final CatBoost model trained.
Selected depth: 12


In [15]:
# Songhao -> Save CatBoost results

results_dir = repo_dir / "data" / "processed" / "catboost_results"
results_dir.mkdir(parents=True, exist_ok=True)

catboost_results_df.to_csv(results_dir / "catboost_outer_cv_results.csv", index=False)
catboost_tuning_df.to_csv(results_dir / "catboost_inner_tuning_results.csv", index=False)
catboost_summary_df.to_csv(results_dir / "catboost_summary.csv", index=False)

print("Saved CatBoost results to:", results_dir)

Saved CatBoost results to: /content/SML_PG60/data/processed/catboost_results


In [19]:
#Lachlan -> autoencoder

class FeatureAutoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()

        hidden_dim = max(256, bottleneck_dim * 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def encode_data(X, new_dim, epochs = 50, batch_size = 32, lr = 0.001):
  #Convert to tensor
  Xt = torch.tensor(X, dtype = torch.float32)
  X_dim = Xt.shape[1]

  dataset = torch.utils.data.TensorDataset(Xt)
  data_loader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True)

  autoencoder_model = FeatureAutoencoder(input_dim = X_dim, bottleneck_dim = new_dim)
  criterion = nn.MSELoss()
  #Use Adam for best performance
  optimizer = torch.optim.Adam(autoencoder_model.parameters(), lr = lr)

  autoencoder_model.train()
  for i in range(epochs):
    for (batch_x,) in data_loader:
      reconstructed_data = autoencoder_model(batch_x)
      loss = criterion(reconstructed_data, batch_x)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  autoencoder = autoencoder_model.encoder
  autoencoder.eval()
  with torch.no_grad():
    reduced_X = autoencoder(Xt)

  new_X_np = reduced_X.detach().cpu().numpy()

  return new_X_np

In [ ]:
#Lachlan -> Determine optimal autoencoding

#X and y must be numpy arrays
#X must be 2D

step_size = 50
def opt_autoencode(k_folds, X, y, epochs = 50, batch_size = 32, lr = 0.001, momentum = 1e-2, step_size):
  X_len = X.shape[1] #Starting length
  test_len = np.floor_divide(X_len, 2)
  best_f1 = 0
  best_len = 0
  best_X = np.empty(X.shape)

  while test_len > 10:
    X_encoded = encode_data(X, test_len, epochs = epochs, batch_size= batch_size, lr = lr)
    criterion = torch.nn.BCEWithLogitsLoss()
    test_f1 = (cross_val(k_folds, X_encoded, y, LogisticRegressionModel,criterion, lr, momentum))[3]
    if test_f1 > best_f1:
      best_len = test_len
      best_X = X_encoded.copy()
    test_len = test_len - 50

  return best_X


In [ ]:
#Lachlan -> Autoencode X
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("influenza_outbreak_long.csv")
df.drop(columns = ['location','split', 'time_index'], inplace = True) #Basic version
df['label'] = df['label'].astype(int) #Cast entries as ints

scaler = StandardScaler()
X_np = scaler.fit_transform((df.drop(columns = ['label'])).values)
y_np = df['label'].values
k_folds = 5
epochs = 30
batch_size = 32
lr = 0.001
momentum = 1e-2

X_reduced = opt_autoencode(k_folds, X_np, y_np, epochs, batch_size, lr, momentum)
print(X_reduced.shape)

In [17]:
#Lachlan -> Copied Amelia's state filtering
c_df = pd.read_csv(repo_dir / "data/covid19_tweets.csv.xz")
import re

c_df["user_location"] = c_df["user_location"].fillna("").astype(str).str.strip().str.lower()
# dictionaries
abbr_to_state = {
    "al": "alabama","ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california","co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon",
    "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina", "sd": "south dakota",
    "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia",
    "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}
state_names = set(abbr_to_state.values())

usa_terms = {"usa", "us", "united states", "america", "u.s.", "u.s.a."}

city_to_state = {
    "new york": "new york", "nyc": "new york", "brooklyn": "new york", "manhattan": "new york",
    "los angeles": "california", "san diego": "california", "san francisco": "california", "sacramento": "california", "long beach": "california",
    "houston": "texas", "dallas": "texas", "austin": "texas",
    "miami": "florida", "orlando": "florida",
    "chicago": "illinois",
    "atlanta": "georgia",
    "boston": "massachusetts",
    "las vegas": "nevada",
    "seattle": "washington",
    "new orleans": "louisiana",
    "st louis": "missouri", "saint louis": "missouri",
    "washington dc": "district of columbia"
}

# specific non-us locations/terms to exclude
junk = ["everywhere", "worldwide", "23 countries", "opt-out", "catch me", "the beach",
        "in the vineyard", "available now", "working"]

foreign = ["canada", "uk", "australia", "india", "germany", "france", "italy", "spain", "brazil",
           "argentina", "china", "japan", "south korea", "paris", "london", "vienna", "delhi", "rio",
           "beijing", "nairobi", "joburg"]

bad_ab = {"in", "or", "me", "hi"}

def is_ambig(loc): # multiple locations or terms foreign to usa
    separators = [",", ";", "|", "/", " and ", " & "]
    foreign_count = 0
    for term in foreign:
        if term in loc:
            foreign_count += 1
    if foreign_count >= 1 and any (sep in loc for sep in separators):
        return True
    return False

def is_junk(loc):
    for term in junk:
        if term in loc:
            return True
    return False

#filter
def get_state(loc):
    loc = loc.lower().strip()

    if loc == "":
        return None

    if is_junk(loc):
        return None

    if is_ambig(loc):
        return None

    for state in state_names:
        if state in loc:
            return state

    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return abbr_to_state[token]

    for city in city_to_state:
        if city in loc:
            return city_to_state[city]
    return None

# confidence levels (for later analysis.. noise filtering, weighting, etc.)
def loc_conf(loc):
    loc = loc.lower().strip()
    for state in state_names:
        if state in loc:
            return "high"
    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return "high"
    for city in city_to_state:
        if city in loc:
            return "medium"
    for term in usa_terms:
        if term in loc:
            return "low"
    return "none"

c_df["state"] = c_df["user_location"].apply(get_state)
c_df["state_confidence"] = c_df["user_location"].apply(loc_conf)
state_cdf = c_df[c_df["state"].notna()].copy()
print(state_cdf[["user_location", "state", "state_confidence"]].head(100))
print(len(state_cdf), "rows with identified US state locations")

             user_location     state state_confidence
1             new york, ny  new york             high
2         pewee valley, ky  kentucky             high
6          gainesville, fl   florida             high
19            florida, usa   florida             high
22       northwest indiana   indiana             high
..                     ...       ...              ...
471     mount prospect, il  illinois             high
473                vermont   vermont             high
474  saint louis, missouri  missouri             high
475           florida, usa   florida             high
476          virginia, usa  virginia             high

[100 rows x 3 columns]
38522 rows with identified US state locations


In [19]:
#Lachlan -> Convert tweet text to dictionary using first dataset keywords
csv_path = repo_dir / "data" / "processed" / "flu_csv" / "sg_flucsv" / "influenza_outbreak_long.csv"
flu_df = pd.read_csv(csv_path)
flu_columns = flu_df.columns.tolist()
columns_to_drop = ['location', 'split', 'time_index', 'label']
flu_keywords = [item for item in flu_columns if item not in columns_to_drop]

def vectorise_tweet(tweet):
  text = tweet.lower()
  counts = {}
  for keyword in flu_keywords:
    count = text.count(keyword)
    counts[keyword] = count
  return counts



In [21]:
#Lachlan -> Sort COVID tweets by whether or not the 'covid19' hashtag appears.

covid_df = pd.read_csv(repo_dir / "data" / "covid19_tweets.csv.xz")
word_count_series = covid_df['text'].apply(vectorise_tweet)
covid_count_df = pd.DataFrame(list(word_count_series))
covid_vector_df = covid_count_df.reindex(columns = flu_df.columns, fill_value = 0)
covid_vector_df.head(20)


,location,split,time_index,flu,swine,stomach,symptoms,virus,bug,strep,...,tests,thinks,ankle,work,hand,complications,children,start,aja,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
# Songhao -> Convert COVID tweets into the same feature space as influenza data

import re
import numpy as np
import pandas as pd

# Use the US-state filtered COVID tweets from the previous state filtering cell
covid_external_df = state_cdf.copy()

def clean_tweet_text(text):
    text = "" if pd.isna(text) else str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def combine_tweet_text(row):
    text = "" if pd.isna(row.get("text", "")) else str(row.get("text", ""))
    hashtags = "" if pd.isna(row.get("hashtags", "")) else str(row.get("hashtags", ""))
    return clean_tweet_text(text + " " + hashtags)


def keyword_pattern(keyword):
    keyword = str(keyword).lower().strip().replace("_", " ")
    keyword = re.sub(r"\s+", " ", keyword)

    if keyword == "":
        return None

    # Avoid matching short words inside longer words, e.g. "flu" inside "influence"
    return r"(?<![a-z0-9])" + re.escape(keyword) + r"(?![a-z0-9])"


def vectorise_tweets_to_keyword_features(tweet_text, keyword_cols):
    feature_dict = {}

    for keyword in keyword_cols:
        pattern = keyword_pattern(keyword)

        if pattern is None:
            feature_dict[keyword] = np.zeros(len(tweet_text), dtype=np.float32)
        else:
            feature_dict[keyword] = (
                tweet_text
                .str.contains(pattern, regex=True, na=False)
                .astype(np.float32)
                .to_numpy()
            )

    feature_df = pd.DataFrame(feature_dict, index=tweet_text.index)
    return feature_df


# Clean and combine tweet text + hashtags
covid_clean_text = covid_external_df.apply(combine_tweet_text, axis=1)

# Convert COVID tweets to influenza keyword features
X_covid_keywords = vectorise_tweets_to_keyword_features(
    covid_clean_text,
    keyword_cols
)

# Add the same non-temporal summary features used in CatBoost training
summary_features = pd.DataFrame(index=X_covid_keywords.index)

summary_features["keyword_total_intensity"] = X_covid_keywords.sum(axis=1)
summary_features["active_keyword_count"] = (X_covid_keywords > 0).sum(axis=1)
summary_features["nonzero_keyword_ratio"] = summary_features["active_keyword_count"] / len(keyword_cols)
summary_features["max_keyword_value"] = X_covid_keywords.max(axis=1)
summary_features["mean_keyword_value"] = X_covid_keywords.mean(axis=1)
summary_features["std_keyword_value"] = X_covid_keywords.std(axis=1)
summary_features["log_total_intensity"] = np.log1p(summary_features["keyword_total_intensity"])

# Combine keyword features and summary features once
X_covid_features = pd.concat(
    [X_covid_keywords, summary_features],
    axis=1
)

# Align exactly with the feature columns used to train CatBoost
X_covid_external = (
    X_covid_features
    .reindex(columns=catboost_feature_columns, fill_value=0)
    .astype(np.float32)
)

print("COVID external data shape:", covid_external_df.shape)
print("COVID external feature shape:", X_covid_external.shape)
print("Matches CatBoost training columns:", list(X_covid_external.columns) == catboost_feature_columns)

X_covid_external.head()

COVID external data shape: (38522, 15)
COVID external feature shape: (38522, 532)
Matches CatBoost training columns: True


,flu,swine,stomach,symptoms,virus,bug,strep,season,influenza,fever,...,children,start,aja,keyword_total_intensity,active_keyword_count,nonzero_keyword_ratio,max_keyword_value,mean_keyword_value,std_keyword_value,log_total_intensity
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,2.0,0.003810,1.0,0.003810,0.061662,1.098612
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.001905,1.0,0.001905,0.043644,0.693147
19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.001905,1.0,0.001905,0.043644,0.693147
22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,4.0,4.0,0.007619,1.0,0.007619,0.087037,1.609438


In [24]:
# Songhao -> Create weak labels for COVID tweets

COVID_PROXY_TERMS = [
    "covid",
    "covid19",
    "covid-19",
    "coronavirus",
    "sars-cov-2",
    "sars cov 2",
]

covid_proxy_pattern = r"(?<![a-z0-9])(" + "|".join(
    re.escape(term) for term in COVID_PROXY_TERMS
) + r")(?![a-z0-9])"

y_covid_weak = (
    covid_clean_text
    .str.contains(covid_proxy_pattern, regex=True, na=False)
    .astype(int)
    .to_numpy()
)

print("COVID weak-label distribution:")
print(pd.Series(y_covid_weak).value_counts())
print("Weak positive rate:", y_covid_weak.mean())

COVID weak-label distribution:
1    23670
0    14852
Name: count, dtype: int64
Weak positive rate: 0.6144540781890867


/tmp/ipykernel_564/399867754.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(covid_proxy_pattern, regex=True, na=False)


In [25]:
# Songhao -> Run COVID external weak-label test for three models

# ============================================================
# Model configuration
# ============================================================
# Replace the placeholder model variables after each final model is trained.
#
# MODEL_FEATURE_TRANSFORM:
# - Use None if the model was trained directly on X.
# - Use fitted_scaler.transform if the model was trained on scaled features.
# - For CatBoost, use None.

external_model_configs = [
    # {
    #     "model_id": "model1",
    #     "model_name": "Model 1 - Logistic Regression",
    #     "model": final_logistic_model,          # change this to your final model 1 variable
    #     "feature_transform": logistic_scaler.transform,  # use None if no scaler was used
    #     "threshold": None,
    # },
    # {
    #     "model_id": "model2",
    #     "model_name": "Model 2 - Random Forest",
    #     "model": final_random_forest_model,     # change this to your final model 2 variable
    #     "feature_transform": None,
    #     "threshold": None,
    # },
    {
        "model_id": "model3",
        "model_name": "Model 3 - CatBoost",
        "model": final_catboost_model,
        "feature_transform": None,
        "threshold": None,
    },
]

In [27]:
# Songhao -> Define generic external weak-label test function

def predict_external_labels(model, X_external, feature_transform=None, threshold=None):
    X_input = X_external

    if feature_transform is not None:
        X_input = feature_transform(X_input)

    if threshold is not None and hasattr(model, "predict_proba"):
        proba = np.asarray(model.predict_proba(X_input))

        if proba.ndim == 2:
            positive_score = proba[:, 1]
        else:
            positive_score = proba

        y_pred = (positive_score >= threshold).astype(int)
        return y_pred, positive_score

    y_pred = np.asarray(model.predict(X_input)).reshape(-1).astype(int)

    positive_score = None
    if hasattr(model, "predict_proba"):
        proba = np.asarray(model.predict_proba(X_input))
        if proba.ndim == 2:
            positive_score = proba[:, 1]

    return y_pred, positive_score


def run_external_weak_label_test(
    model,
    model_name,
    X_external,
    y_weak,
    covid_df,
    cleaned_text,
    feature_transform=None,
    threshold=None,
):
    y_pred, positive_score = predict_external_labels(
        model=model,
        X_external=X_external,
        feature_transform=feature_transform,
        threshold=threshold,
    )

    metrics = compute_metrics(y_weak, y_pred)

    results_df = pd.DataFrame([{
        "model": model_name,
        "weak_label_accuracy": metrics["accuracy"],
        "weak_label_precision": metrics["precision"],
        "weak_label_recall": metrics["recall"],
        "weak_label_f1": metrics["f1"],
        "tp": metrics["tp"],
        "tn": metrics["tn"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "weak_positive_rate": float(np.mean(y_weak)),
        "model_positive_rate": float(np.mean(y_pred)),
        "n_samples": len(y_weak),
    }])

    predictions_df = covid_df.copy()
    predictions_df["cleaned_text"] = cleaned_text.values
    predictions_df["covid_weak_label"] = y_weak
    predictions_df[f"{model_name}_prediction"] = y_pred

    if positive_score is not None:
        predictions_df[f"{model_name}_positive_score"] = positive_score

    return results_df, predictions_df

In [28]:
# Songhao -> Run COVID weak-label external test for all configured models -- domain transfer test

all_external_results = []
all_external_predictions = {}

for config in external_model_configs:
    print(f"Running external test for {config['model_id']}: {config['model_name']}")

    results_df, predictions_df = run_external_weak_label_test(
        model=config["model"],
        model_name=config["model_name"],
        X_external=X_covid_external,
        y_weak=y_covid_weak,
        covid_df=covid_external_df,
        cleaned_text=covid_clean_text,
        feature_transform=config["feature_transform"],
        threshold=config["threshold"],
    )

    results_df.insert(0, "model_id", config["model_id"])
    all_external_results.append(results_df)
    all_external_predictions[config["model_id"]] = predictions_df

covid_external_results_all_df = pd.concat(
    all_external_results,
    ignore_index=True,
)

# Reorder columns for clearer display
display_cols = [
    "model_id",
    "model",
    "weak_label_accuracy",
    "weak_label_precision",
    "weak_label_recall",
    "weak_label_f1",
    "tp",
    "tn",
    "fp",
    "fn",
    "weak_positive_rate",
    "model_positive_rate",
    "n_samples",
]

covid_external_results_all_df = covid_external_results_all_df[display_cols]

display(covid_external_results_all_df)

Running external test for model3: Model 3 - CatBoost


,model_id,model,weak_label_accuracy,weak_label_precision,weak_label_recall,weak_label_f1,tp,tn,fp,fn,weak_positive_rate,model_positive_rate,n_samples
0,model3,Model 3 - CatBoost,0.385572,1.0,0.000042,0.000084,1,14852,0,23669,0.614454,0.000026,38522


In [29]:
# Songhao -> Save COVID weak-label external test results

external_results_dir = repo_dir / "data" / "processed" / "external_covid_test"
external_results_dir.mkdir(parents=True, exist_ok=True)

# 1. Save combined metric summary for all models
covid_external_results_all_df.to_csv(
    external_results_dir / "all_models_covid_weak_label_results.csv",
    index=False,
)

# 2. Save each model's prediction details separately
for config in external_model_configs:
    model_id = config["model_id"]
    model_name = config["model_name"]

    # Save one-row metric result for this model
    model_result_df = covid_external_results_all_df[
        covid_external_results_all_df["model_id"] == model_id
    ]

    model_result_df.to_csv(
        external_results_dir / f"{model_id}_covid_weak_label_metrics.csv",
        index=False,
    )

    # Save prediction details for this model
    all_external_predictions[model_id].to_csv(
        external_results_dir / f"{model_id}_covid_weak_label_predictions.csv",
        index=False,
    )

    print(f"Saved results for {model_id}: {model_name}")

print("All COVID external test results saved to:", external_results_dir)

Saved results for model3: Model 3 - CatBoost
All COVID external test results saved to: /content/SML_PG60/data/processed/external_covid_test


In [32]:
pd.read_csv(external_results_dir / "all_models_covid_weak_label_results.csv")

,model_id,model,weak_label_accuracy,weak_label_precision,weak_label_recall,weak_label_f1,tp,tn,fp,fn,weak_positive_rate,model_positive_rate,n_samples
0,model3,Model 3 - CatBoost,0.385572,1.0,0.000042,0.000084,1,14852,0,23669,0.614454,0.000026,38522
